# Normalização dos dados e camada processed

Transforma os dados brutos de `data/raw/` em uma camada tratada em `data/processed/`, que será a base comum para todas as análises do grupo.

Etapas:
1. Converter as colunas de data para o tipo data.
2. Padronizar os textos: maiúsculas, sem acentos, sem espaços nas pontas e sem espaços duplos.
3. Remover as linhas duplicadas da geolocalização.
4. Criar indicadores no nível do pedido: valores, atraso, avaliação, localização do cliente e mês da compra.
5. Salvar as tabelas tratadas.

A única remoção de linhas é a de duplicatas exatas da geolocalização, que não trazem informação nova. Todos os pedidos, inclusive cancelados e não entregues, são mantidos: o filtro por status é aplicado em cada análise conforme a necessidade (ex.: receita considera apenas pedidos `DELIVERED`).

## Bibliotecas

In [ ]:
%run ../src/librarys.ipynb

## Carga dos dados brutos

Mesma leitura do inventário: CEPs como texto (para manter o zero à esquerda) e `utf-8-sig` na tabela de tradução (por causa do caractere invisível no início do arquivo).

In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv", encoding="utf-8-sig")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})

## 1. Conversão das datas

As datas foram lidas como texto. Convertê-las para o tipo data permite calcular diferenças entre elas (ex.: dias de atraso) e agrupar por mês. Isso é feito antes da padronização de textos para que as datas não sejam tratadas como texto.

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

orders.dtypes

## 2. Padronização dos textos

Textos escritos de formas diferentes (`sao paulo`, `Sao Paulo`, `sao  paulo`) seriam tratados como valores distintos em agrupamentos e joins. A função abaixo aplica a mesma regra a todas as colunas de texto de uma tabela:

- converte para **maiúsculas**;
- remove **espaços no início e no fim**;
- troca **dois ou mais espaços seguidos** por um só.

As colunas de identificador (terminadas em `_id`) ficam de fora: são códigos, não textos, e precisam continuar idênticos aos dados originais.

In [ ]:
def normalizar_texto(df):
    for coluna in df.select_dtypes(include="str").columns:
        if coluna.endswith("_id"):
            continue
        df[coluna] = (
            df[coluna]
            .str.upper()
            .str.normalize("NFKD")
            .str.encode("ascii", "ignore")
            .str.decode("ascii")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    return df

Exemplo antes da padronização: cidades da geolocalização com espaços duplos ou espaço no fim do nome.

In [ ]:
geolocation[geolocation["geolocation_city"].str.contains("  ") | geolocation["geolocation_city"].str.endswith(" ")]["geolocation_city"].value_counts().head(10)

Exemplo antes da padronização: a mesma cidade escrita com e sem acento.

In [ ]:
geolocation[geolocation["geolocation_city"].isin(["sao paulo", "são paulo"])]["geolocation_city"].value_counts()

Aplica a padronização em todas as tabelas.

In [ ]:
customers = normalizar_texto(customers)
orders = normalizar_texto(orders)
order_items = normalizar_texto(order_items)
payments = normalizar_texto(payments)
reviews = normalizar_texto(reviews)
products = normalizar_texto(products)
category_translation = normalizar_texto(category_translation)
sellers = normalizar_texto(sellers)
geolocation = normalizar_texto(geolocation)

Resultado da padronização em algumas tabelas.

In [ ]:
geolocation[geolocation["geolocation_city"] == "SAO PAULO"]["geolocation_city"].value_counts()

In [ ]:
customers.head(10)

In [ ]:
orders["order_status"].value_counts()

In [ ]:
products.head(10)

## 3. Remoção de duplicatas da geolocalização

A geolocalização tem linhas inteiramente repetidas (mesmo CEP, coordenadas, cidade e UF). Após a padronização dos textos, linhas que diferiam só por acento ou maiúscula (`são paulo` / `SAO PAULO`) também passam a ser iguais. Essas linhas não trazem informação nova e só aumentam o tamanho do arquivo.

Os demais registros são mantidos: um mesmo CEP continua com várias coordenadas diferentes, que poderão ser consolidadas (ex.: pela média) quando alguém for usar mapas.

In [ ]:
linhas_antes = len(geolocation)
geolocation = geolocation.drop_duplicates()

print("Linhas antes:", linhas_antes)
print("Linhas depois:", len(geolocation))
print("Removidas:", linhas_antes - len(geolocation))

## 4. Indicadores do pedido

Cria uma tabela de pedidos enriquecida (`orders_enriched`), com uma linha por pedido e as colunas abaixo:

| Coluna | Descrição |
|---|---|
| `order_price` | Soma do preço dos itens do pedido |
| `order_freight` | Soma do frete dos itens do pedido |
| `order_total` | Valor total do pedido: preço + frete |
| `review_score` | Média das avaliações do pedido |
| `customer_state` | UF do cliente |
| `customer_city` | Cidade do cliente |
| `purchase_year_month` | Ano e mês da compra (ex.: `2017-11`), para agregações mensais |
| `dt_diff` | Dias entre a data de entrega e a data estimada. Positivo = atrasou; negativo = chegou antes |
| `is_late` | 1 se o pedido foi entregue com atraso, 0 se no prazo |

### `order_price`, `order_freight` e `order_total`

Um pedido pode ter vários itens. Somamos preço e frete de todos os itens de cada pedido. Os dois valores são mantidos separados, porque a análise de frete precisa deles, e somados em `order_total`.

In [ ]:
total_por_pedido = order_items.groupby("order_id")[["price", "freight_value"]].sum().reset_index()
total_por_pedido.columns = ["order_id", "order_price", "order_freight"]
total_por_pedido["order_total"] = total_por_pedido["order_price"] + total_por_pedido["order_freight"]

total_por_pedido.head(10)

### `review_score`

Alguns pedidos têm mais de uma avaliação (visto no inventário). Usamos a média para ficar com um único valor por pedido.

In [ ]:
review_por_pedido = reviews.groupby("order_id")["review_score"].mean().reset_index()
review_por_pedido.head(10)

### Junção com pedidos

Junta pedidos com o total, a avaliação e a cidade/UF do cliente. Usamos `how="left"` para manter **todos os pedidos**, mesmo os que não têm itens ou avaliação (nesses casos a coluna fica vazia).

In [ ]:
orders_enriched = orders.merge(total_por_pedido, on="order_id", how="left")
orders_enriched = orders_enriched.merge(review_por_pedido, on="order_id", how="left")
orders_enriched = orders_enriched.merge(customers[["customer_id", "customer_state", "customer_city"]], on="customer_id", how="left")

print("Pedidos antes:", len(orders))
print("Pedidos depois:", len(orders_enriched))

### `purchase_year_month`

Ano e mês da compra no formato `AAAA-MM`. Facilita as agregações mensais (evolução de pedidos, receita e ticket médio) sem precisar converter a data em cada análise.

In [ ]:
orders_enriched["purchase_year_month"] = orders_enriched["order_purchase_timestamp"].dt.strftime("%Y-%m")

orders_enriched["purchase_year_month"].value_counts().sort_index()

### `dt_diff` e `is_late`

`dt_diff` é a diferença em dias entre a entrega real e a data estimada. Comparamos apenas as datas (sem o horário), porque a data estimada não tem hora: um pedido entregue às 15h do dia previsto está no prazo.

`is_late` vale 1 quando `dt_diff` é maior que zero. Pedidos que **não foram entregues** (em trânsito, cancelados) ficam vazios nas duas colunas: não é possível dizer se atrasaram.

In [ ]:
data_entrega = orders_enriched["order_delivered_customer_date"].dt.normalize()
data_estimada = orders_enriched["order_estimated_delivery_date"].dt.normalize()

orders_enriched["dt_diff"] = (data_entrega - data_estimada).dt.days

orders_enriched["is_late"] = (orders_enriched["dt_diff"] > 0).astype("Int64")
orders_enriched.loc[orders_enriched["dt_diff"].isna(), "is_late"] = pd.NA

Conferência: quantidade de pedidos atrasados, no prazo e sem entrega.

In [ ]:
orders_enriched["is_late"].value_counts(dropna=False)

Distribuição de `dt_diff`. A mediana negativa indica que a maior parte dos pedidos chega antes da data estimada.

In [ ]:
orders_enriched["dt_diff"].describe()

Resultado final da tabela de pedidos enriquecida.

In [ ]:
orders_enriched[[
    "order_id", "order_status", "order_purchase_timestamp", "purchase_year_month",
    "order_delivered_customer_date", "order_estimated_delivery_date", "dt_diff", "is_late",
    "order_price", "order_freight", "order_total", "review_score", "customer_state", "customer_city",
]].head(10)

In [ ]:
orders_enriched[["order_price", "order_freight", "order_total", "review_score", "dt_diff", "is_late"]].isna().sum()

## 5. Salvar a camada processed

Grava as tabelas padronizadas e a tabela de pedidos enriquecida em `data/processed/`. Os demais notebooks do projeto devem ler daqui, e não de `data/raw/`.

In [ ]:
customers.to_csv("../data/processed/customers.csv", index=False)
orders.to_csv("../data/processed/orders.csv", index=False)
order_items.to_csv("../data/processed/order_items.csv", index=False)
payments.to_csv("../data/processed/payments.csv", index=False)
reviews.to_csv("../data/processed/reviews.csv", index=False)
products.to_csv("../data/processed/products.csv", index=False)
category_translation.to_csv("../data/processed/category_translation.csv", index=False)
sellers.to_csv("../data/processed/sellers.csv", index=False)
geolocation.to_csv("../data/processed/geolocation.csv", index=False)

orders_enriched.to_csv("../data/processed/orders_enriched.csv", index=False)